# WAXAL-Dual-Core: End-to-End Training Pipeline

This notebook implements the complete pipeline for eliminating the Tokenization Tax
for African languages via dual-stream tokenization.

**Target Hardware:**
- Training: Google Colab T4 GPU
- Deployment: Dell Latitude 7400 (8GB RAM)

**Target Language:** Akan (proof-of-concept)

**Pipeline:**
1. Setup & Dependencies
2. Download WAXAL Dataset
3. Train BPE Vocabularies (ASR + TTS)
4. Train LoRA Variants
5. Benchmark Token Fertility
6. Export Results

## 1. Setup & Dependencies

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers peft bitsandbytes datasets accelerate
!pip install -q torch sentencepiece tokenizers
!pip install -q llama-cpp-python psutil

In [ ]:
# Clone repository (if running on Colab)
import os
from pathlib import Path

if not os.path.exists('project-soma'):
    !git clone https://github.com/attabeezy/project-soma.git
    %cd project-soma
else:
    %cd project-soma

In [ ]:
import os
import json
import sys
import shutil
from pathlib import Path

# Set HuggingFace token (required for Llama models)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. You may need to set it for Llama models.")
    print("Set it with: os.environ['HF_TOKEN'] = 'your_token'")

In [ ]:
# Setup Google Drive auto-save
def setup_drive_save():
    """Mount Google Drive and return destination path."""
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_ms=120000)
        DRIVE_BASE = '/content/drive/MyDrive/WAXAL-results'
        os.makedirs(DRIVE_BASE, exist_ok=True)
        print(f"Google Drive mounted: {DRIVE_BASE}")
        return DRIVE_BASE
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str, step_name: str):
    """Save a directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest_dir = os.path.join(drive_base, step_name)
    try:
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)
        shutil.copytree(source_dir, dest_dir)
        print(f"Saved {step_name} to Google Drive: {dest_dir}")
    except Exception as e:
        print(f"Error saving {step_name}: {e}")

# Initialize Drive
DRIVE_BASE = setup_drive_save()

## 2. Download WAXAL Dataset

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

# Configuration
LANGUAGE = "akan"  # Proof-of-concept
ASR_CONFIG = "aka_asr"  # Akan ASR
TTS_CONFIG = "twi_tts"  # Twi TTS

# Language config map
LANGUAGE_CONFIGS = {
    "akan": {"asr": "aka_asr", "tts": "twi_tts"},
    "yoruba": {"asr": None, "tts": "yor_tts"},
    "swahili": {"asr": None, "tts": "swa_tts"},
}

config = LANGUAGE_CONFIGS[LANGUAGE]
print(f"Downloading WAXAL dataset for {LANGUAGE}...")

In [ ]:
def download_and_save_dataset(config_name: str, split: str, output_dir: Path):
    """Download dataset and save as JSONL."""
    print(f"Downloading {config_name}/{split}...")
    
    dataset = load_dataset(
        "google/WaxalNLP",
        config_name,
        split=split,
        trust_remote_code=True
    )
    
    output_file = output_dir / f"{config_name}_{split}.jsonl"
    
    with open(output_file, "w", encoding="utf-8") as f:
        for item in tqdm(dataset):
            item_dict = {
                "id": item["id"],
                "speaker_id": item.get("speaker_id", ""),
                "transcription": item.get("transcription", item.get("text", "")),
                "language": item.get("language", item.get("locale", "")),
                "gender": item.get("gender", ""),
            }
            f.write(json.dumps(item_dict, ensure_ascii=False) + "\n")
    
    print(f"Saved {len(dataset)} samples to {output_file}")
    return len(dataset)

In [ ]:
# Create data directory
data_dir = Path("data") / LANGUAGE
data_dir.mkdir(parents=True, exist_ok=True)

# Download ASR data
asr_samples = 0
if config["asr"]:
    for split in ["train", "validation", "test"]:
        try:
            asr_samples += download_and_save_dataset(config["asr"], split, data_dir)
        except Exception as e:
            print(f"  Error downloading {config['asr']}/{split}: {e}")

# Download TTS data
tts_samples = 0
if config["tts"]:
    for split in ["train", "validation", "test"]:
        try:
            tts_samples += download_and_save_dataset(config["tts"], split, data_dir)
        except Exception as e:
            print(f"  Error downloading {config['tts']}/{split}: {e}")

print("\nDownloaded {asr_samples} ASR samples and {tts_samples} TTS samples")

In [ ]:
# Save dataset to Google Drive
save_to_drive(str(data_dir), DRIVE_BASE, "data")
print("Dataset saved to Drive!")

In [ ]:
# Save metadata
metadata = {
    "language": LANGUAGE,
    "asr_config": config["asr"],
    "tts_config": config["tts"],
    "asr_samples": asr_samples,
    "tts_samples": tts_samples,
}

with open(data_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved!")

## 3. Baseline Token Fertility Analysis

In [ ]:
from transformers import AutoTokenizer
import statistics

# Load baseline tokenizer (Llama-3.2-1B)
BASE_MODEL = "meta-llama/Llama-3.2-1B"

print(f"Loading baseline tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load test texts
def load_texts(jsonl_path: Path, max_samples: int = 100) -> list[str]:
    texts = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            item = json.loads(line.strip())
            text = item.get("transcription", "")
            if text:
                texts.append(text)
    return texts

# Calculate baseline fertility
test_file = data_dir / f"{config['tts']}_test.jsonl"
test_texts = load_texts(test_file, max_samples=100)

print(f"\nAnalyzing {len(test_texts)} test samples...")

fertilites = []
for text in test_texts:
    words = len(text.split())
    tokens = len(tokenizer.encode(text))
    if words > 0:
        fertilites.append(tokens / words)

baseline_fertility = statistics.mean(fertilites)
print(f"\n=== BASELINE Token Fertility ===")
print(f"Mean: {baseline_fertility:.2f} tokens/word")
print(f"Min: {min(fertilites):.2f}")
print(f"Max: {max(fertilites):.2f}")
print(f"Target: {baseline_fertility * 0.7:.2f} (30% reduction)")

## 4. Train BPE Vocabularies

In [ ]:
# Run BPE training script
!python scripts/train_bpe.py \
    --input {data_dir} \
    --output models/tokenizers/ \
    --language {LANGUAGE} \
    --vocab-size 8000

## 5. Train LoRA Variants

In [ ]:
# Configure training
TRAIN_GROUP = "E"  # Primary hypothesis: TTS -> ASR -> TTS
BASE_MODEL = "meta-llama/Llama-3.2-1B"

# For Colab T4, we need to authenticate with HuggingFace for Llama models
# Uncomment and set your token:
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

In [ ]:
# Run LoRA training (Variant E)
# Note: This may take several hours on T4
# Reduce epochs for testing:

!python scripts/train_lora.py \
    --group {TRAIN_GROUP} \
    --model {BASE_MODEL} \
    --data {data_dir} \
    --output checkpoints/ \
      --epochs 1

In [ ]:
# Save checkpoints to Google Drive
save_to_drive('checkpoints', DRIVE_BASE, "checkpoints")
print("Checkpoints saved to Drive!")

## 6. Benchmark Results

In [ ]:
# Benchmark baseline vs trained
import subprocess
import json

# Baseline benchmark
print("=== BASELINE BENCHMARK ===")
!python benchmark.py \
    --tokenizer {BASE_MODEL} \
    --test-file {test_file} \
    --baseline \
    --output results/baseline.json

In [ ]:
# Trained model benchmark (if training completed)
checkpoint_dir = Path(f"checkpoints/variant_{TRAIN_GROUP}/final")

if checkpoint_dir.exists():
    print("=== TRAINED MODEL BENCHMARK ===")
    !python benchmark.py \
        --model {checkpoint_dir} \
        --test-file {test_file} \
        --huggingface \
        --output results/variant_{TRAIN_GROUP}.json

    # Compare results
    baseline = json.load(open("results/baseline.json"))
    trained = json.load(open(f"results/variant_{TRAIN_GROUP}.json"))
    
    reduction = (baseline["fertility"] - trained["fertility"]) / baseline["fertility"] * 100
    
    print(f"\n=== COMPARISON ===")
    print(f"Baseline fertility: {baseline['fertility']:.2f}")
    print(f"Trained fertility: {trained['fertility']:.2f}")
    print(f"Reduction: {reduction:.1f}%")
    print(f"Target: 30%")
    
    if reduction >= 30:
        print("\nTARGET ACHIEVED")
    else:
        print("\nTarget not reached")
else:
    print(f"Checkpoint not found at {checkpoint_dir}")
    print("Training may not have completed. Results will be available after training.")

## 7. Export to GGUF (for Edge Deployment)

In [ ]:
# Export to GGUF format
if checkpoint_dir.exists():
    print("Exporting to GGUF format...")
    !python scripts/export_gguf.py \
        --checkpoint {checkpoint_dir} \
        --output models/gguf/ \
        --quantization Q4_K_M
else:
    print("Skipping GGUF export - checkpoint not found")

In [ ]:
# Save final results to Google Drive
save_to_drive('results', DRIVE_BASE, "results")
if os.path.exists('models'):
    save_to_drive('models', DRIVE_BASE, "models")
print("All results saved to Google Drive!")

## 8. Summary

In [ ]:
# Mount Google Drive and save results
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    results_dest = '/content/drive/MyDrive/WAXAL-results'
    os.makedirs(results_dest, exist_ok=True)
    
    # Copy results
    if os.path.exists('results'):
        shutil.copytree('results', f'{results_dest}/results', dirs_exist_ok=True)
    if os.path.exists('checkpoints'):
        shutil.copytree('checkpoints', f'{results_dest}/checkpoints', dirs_exist_ok=True)
    if os.path.exists('models'):
        shutil.copytree('models', f'{results_dest}/models', dirs_exist_ok=True)
    
    print(f"Results saved to {results_dest}")
except ImportError:
    print("Not running in Colab - skipping Google Drive sync")

## Summary

In [ ]:
print("\n" + "="*50)
print("WAXAL-Dual-Core Pipeline Complete!")
print("="*50)
print(f"\nLanguage: {LANGUAGE}")
print(f"Training group: {TRAIN_GROUP}")
print(f"ASR samples: {asr_samples}")
print(f"TTS samples: {tts_samples}")
print(f"\nBaseline token fertility: {baseline_fertility:.2f}")

if checkpoint_dir.exists():
    trained = json.load(open(f"results/variant_{TRAIN_GROUP}.json"))
    reduction = (baseline_fertility - trained['fertility']) / baseline_fertility * 100
    print(f"Trained token fertility: {trained['fertility']:.2f}")
    print(f"Reduction: {reduction:.1f}%")
    
print("\nNext steps:")
print("1. Download results from Google Drive")
print("2. Run benchmark.py locally on Dell Latitude 7400")
print("3. Compare edge performance with cloud results")